In [2]:
#감정추출
!pip install -U transformers==4.41.2 sentence-transformers==2.7.0 accelerate

In [3]:
from transformers import pipeline
import torch

print(torch.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())

emotion_pipe = pipeline(
    "text-classification",
    model="taehoon222/korean-emotion-classifier-final",
    tokenizer="taehoon222/korean-emotion-classifier-final",
    device=0 if torch.cuda.is_available() else -1,
    top_k=None
)

print("성공")

2.10.0+cu128
GPU 사용 가능: True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

성공


In [5]:
# 사용자별 감정 결과 저장
user_emotions = {}


def extract_emotion(user_id, text):
    # 입력받은 글에서 감정 추출
    result = emotion_pipe(text)[0]

    # 가장 높은 감정 선택
    best = max(result, key=lambda x: x["score"])

    # 사용자별 저장
    user_emotions[user_id] = {
        "label": best["label"],
        "score": best["score"],
        "full_result": result
    }

    print(f"\n[{user_id}] 감정 추출 완료")
    print(f"대표 감정: {best['label']} ({best['score']:.4f})")

    print("\n전체 감정 점수:")
    for r in result:
        print(f"- {r['label']}: {r['score']:.4f}")


while True:
    user_id = input("\n사용자 ID 입력 (q 입력 시 종료): ")

    if user_id == "q":
        break

    text = input("편지 입력: ")

    extract_emotion(user_id, text)


# -------------------------------
# 백엔드에서 사용하는 형태 확인
# -------------------------------

print("\n현재 user_emotions:")
print(user_emotions)


사용자 ID 입력 (q 입력 시 종료): 1
편지 입력: 요즘은 하루 끝나고 혼자 산책하는 시간이 제일 좋은 것 같아. 이어폰 끼고 음악 들으면서 천천히 걷다 보면 복잡했던 생각들이 조금씩 정리되는 느낌이 들더라. 시끄러운 분위기보다는 조용하고 차분한 시간을 더 좋아하는 편이야.


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



[1] 감정 추출 완료
대표 감정: 기쁨 (0.9944)

전체 감정 점수:
- 기쁨: 0.9944
- 당황: 0.0020
- 상처: 0.0011
- 불안: 0.0011
- 슬픔: 0.0011
- 분노: 0.0003

사용자 ID 입력 (q 입력 시 종료): 2
편지 입력: 최근에 운동 시작했는데 생각보다 재밌더라. 몸 쓰고 나면 스트레스가 좀 풀리는 느낌이라 계속 하게 되는 것 같아. 운동 끝나고 친구들이랑 맛있는 거 먹으면서 이야기하는 시간도 은근 힐링이야.

[2] 감정 추출 완료
대표 감정: 기쁨 (0.9986)

전체 감정 점수:
- 기쁨: 0.9986
- 불안: 0.0004
- 슬픔: 0.0003
- 분노: 0.0003
- 상처: 0.0002
- 당황: 0.0001

사용자 ID 입력 (q 입력 시 종료): 3
편지 입력: 평소에는 카페 가서 공부하는 걸 좋아해. 집에서는 집중이 잘 안 되는데 카페 특유의 잔잔한 소음 들으면서 하면 오히려 집중이 더 잘 되는 느낌이야. 공부하다가 힘들면 디저트 하나 먹는 걸 작은 보상처럼 생각하고 있어.

[3] 감정 추출 완료
대표 감정: 기쁨 (0.9982)

전체 감정 점수:
- 기쁨: 0.9982
- 분노: 0.0006
- 슬픔: 0.0004
- 불안: 0.0004
- 상처: 0.0003
- 당황: 0.0001

사용자 ID 입력 (q 입력 시 종료): ㅂ
편지 입력: ㅂ

[ㅂ] 감정 추출 완료
대표 감정: 기쁨 (0.5596)

전체 감정 점수:
- 기쁨: 0.5596
- 분노: 0.1492
- 슬픔: 0.1439
- 불안: 0.0987
- 상처: 0.0313
- 당황: 0.0172

사용자 ID 입력 (q 입력 시 종료): q

현재 user_emotions:
{'1': {'label': '기쁨', 'score': 0.9944451451301575, 'full_result': [{'label': '기쁨', 'score': 0.9944451451301575}, {'label': '당황', 'score':